In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install xgboost lightgbm

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/GD/glaucoma_features_CLINICAL_LABEL.csv"

df = pd.read_csv(path)

print("Dataset Loaded Successfully")
print(df.head())

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df["label"].value_counts())

In [ ]:
import seaborn as sns
sns.countplot(x=df["label"])

In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np # Needed for array operations
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# --- Replicating dependencies for X_selected and y ---

# Ensure df is defined. If running this cell out of order, it might not be.
# The notebook state indicates df is available, but if not, load it.
if 'df' not in locals() and 'df' not in globals():
    path = "/content/drive/MyDrive/GD/glaucoma_features_CLINICAL_LABEL.csv"
    df = pd.read_csv(path)

# Apply column stripping if not already applied (from cell ntAkyOgt5vqe)
df.columns = df.columns.str.strip()

# From cell TvGaAZog9hDT
X = df.drop(["dataset","image_index","label","VCDR"], axis=1)
y = df["label"]

# From cell e8nAf7gI_g9W
feature_names = X.columns
X = X.values
y = y.values

# From cell Q12JE91P_GlC (definition of dragonfly_feature_selection)
def dragonfly_feature_selection(X_input, y_input, population_size=20, iterations=30):
    n_features = X_input.shape[1]
    population = np.random.randint(0, 2, (population_size, n_features))
    best_solution = None
    best_score = -1.0 # Initialize with a value lower than any possible accuracy score (0 to 1)

    for iteration in range(iterations):
        fitness = []
        for dragonfly in population:
            if np.sum(dragonfly) < 4: # Force minimum 4 features
                fitness.append(0)
                continue
            selected = np.where(dragonfly == 1)[0]
            # Ensure at least one feature is selected by the dragonfly, or append 0 fitness
            if len(selected) == 0:
                fitness.append(0)
                continue

            X_subset = X_input[:, selected]
            model = RandomForestClassifier(n_estimators=100)
            score = cross_val_score(
                model,
                X_subset,
                y_input,
                cv=5,
                scoring="accuracy"
            ).mean()
            fitness.append(score)

        fitness = np.array(fitness)
        best_idx = np.argmax(fitness)
        if fitness[best_idx] > best_score:
            best_score = fitness[best_idx]
            best_solution = population[best_idx].copy()

        # Dragonfly update rule
        # best_solution is guaranteed not to be None here due to best_score initialization to -1.0
        for i in range(population_size):
            step = best_solution - population[i]
            sigmoid = 1 / (1 + np.exp(-step))
            rand = np.random.rand(n_features)
            population[i] = np.where(rand < sigmoid, 1, 0)

    selected_features_indices = np.where(best_solution == 1)[0]
    return selected_features_indices

# From cell 1ceC38Q4_GoX
selected_features_indices = dragonfly_feature_selection(X, y)
X_selected = X[:, selected_features_indices]

# --- End of replicating dependencies ---

# Apply SMOTE (original code)
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_selected, y)

# Convert to pandas Series for counting (original code)
print("Before balancing:", pd.Series(y).value_counts())
print("After balancing:", pd.Series(y_balanced).value_counts())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x=y_balanced)

plt.title("Balanced Dataset Distribution")

plt.show()

In [ ]:
df.columns=df.columns.str.strip()

In [ ]:
print(df.columns.tolist())

In [ ]:
X = df.drop(["dataset","image_index","label","VCDR"], axis=1)

y = df["label"]

print("Features:", X.columns)
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Features standardized successfully")

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier


def dragonfly_feature_selection(X, y, population_size=20, iterations=30):

    n_features = X.shape[1]

    # Initialize population
    population = np.random.randint(0, 2, (population_size, n_features))

    best_solution = None
    best_score = 0

    for iteration in range(iterations):

        fitness = []

        for dragonfly in population:

            # Ensure at least one feature selected
            if np.sum(dragonfly) == 0:
                fitness.append(0)
                continue

            selected = np.where(dragonfly == 1)[0]
            X_subset = X[:, selected]

            model = RandomForestClassifier(n_estimators=100)

            score = cross_val_score(
                model,
                X_subset,
                y,
                cv=5,
                scoring="accuracy"
            ).mean()

            fitness.append(score)

        fitness = np.array(fitness)

        # Update best solution
        best_idx = np.argmax(fitness)

        if fitness[best_idx] > best_score:
            best_score = fitness[best_idx]
            best_solution = population[best_idx].copy()

        # Update population (Dragonfly movement)
        for i in range(population_size):

            step = best_solution - population[i]

            sigmoid = 1 / (1 + np.exp(-step))

            rand = np.random.rand(n_features)

            population[i] = np.where(rand < sigmoid, 1, 0)

    selected_features = np.where(best_solution == 1)[0]

    return selected_features

In [ ]:
# Save feature names first
feature_names = X.columns

# Convert to numpy arrays
X = X.values
y = y.values

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

def dragonfly_feature_selection(X, y, population_size=20, iterations=30):

    n_features = X.shape[1]

    population = np.random.randint(0, 2, (population_size, n_features))

    best_solution = None
    best_score = 0

    for iteration in range(iterations):

        fitness = []

        for dragonfly in population:

            # Force minimum 4 features
            if np.sum(dragonfly) < 4:
                fitness.append(0)
                continue

            selected = np.where(dragonfly == 1)[0]
            X_subset = X[:, selected]

            model = RandomForestClassifier(n_estimators=100)

            score = cross_val_score(
                model,
                X_subset,
                y,
                cv=5,
                scoring="accuracy"
            ).mean()

            fitness.append(score)

        fitness = np.array(fitness)

        best_idx = np.argmax(fitness)

        if fitness[best_idx] > best_score:
            best_score = fitness[best_idx]
            best_solution = population[best_idx].copy()

        # Dragonfly update rule
        for i in range(population_size):

            step = best_solution - population[i]

            sigmoid = 1 / (1 + np.exp(-step))

            rand = np.random.rand(n_features)

            population[i] = np.where(rand < sigmoid, 1, 0)

    selected_features = np.where(best_solution == 1)[0]

    return selected_features

In [ ]:
selected_features = dragonfly_feature_selection(X, y)

X_selected = X[:, selected_features]

print("Selected feature indices:", selected_features)

print("Selected feature names:")
print(feature_names[selected_features])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),
    "LightGBM": LGBMClassifier(random_state=42)
}

In [ ]:
def evaluate_models(X, y, models, k):

    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    results = []

    for name, model in models.items():

        acc_list = []
        prec_list = []
        rec_list = []
        f1_list = []

        for train_idx, test_idx in kf.split(X):

            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            model.fit(X_train, y_train)
            preds = model.predict(X_test)

            acc_list.append(accuracy_score(y_test, preds))
            prec_list.append(precision_score(y_test, preds))
            rec_list.append(recall_score(y_test, preds))
            f1_list.append(f1_score(y_test, preds))

        results.append([
            name,
            np.mean(acc_list),
            np.mean(prec_list),
            np.mean(rec_list),
            np.mean(f1_list)
        ])

    results_df = pd.DataFrame(results,
                              columns=["Model","Accuracy","Precision","Recall","F1"])

    return results_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier # Removed AdaBoostClassifier
# SVM and KNN are removed based on previous user requests
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
# CatBoost is removed based on user request
# from catboost import CatBoostClassifier

# Train-test split (using X_balanced and y_balanced from previous steps)
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced,
    y_balanced,
    test_size=0.3,
    random_state=42
)

# Define models with the updated list as per user's request
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),
    "LightGBM": LGBMClassifier(random_state=42)
    # Removed AdaBoostClassifier
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    logloss = log_loss(y_test, y_prob)

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        auc,
        logloss
    ])

# Create dataframe
metrics_table = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "AUC Score",
        "Log Loss"
    ]
)

# Print only the requested metrics
print(metrics_table[["Model", "Accuracy", "Precision", "Recall", "F1 Score"]])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

In [ ]:
# =========================
# Import libraries
# =========================
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc

from sklearn.ensemble import RandomForestClassifier # Removed AdaBoostClassifier
# from sklearn.svm import SVC # Removed as per request

# from sklearn.neighbors import KNeighborsClassifier # Removed as per request
from sklearn.tree import DecisionTreeClassifier
# from sklearn.linear_model import LogisticRegression # Removed as per user request

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier # Added as per request
# from catboost import CatBoostClassifier # Removed as per request


# =========================
# Train Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# =========================
# Define Models
# =========================
models = {

    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),

    # "SVM": SVC(probability=True), # Removed as per request

    # "KNN": KNeighborsClassifier(), # Removed as per request

    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),

    "LightGBM": LGBMClassifier(random_state=42) # Added as per request

    # Removed AdaBoostClassifier

}


# =========================
# Plot ROC Curve
# =========================
plt.figure(figsize=(8,6))

for name, model in models.items():

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:,1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")


plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Glaucoma Classification")

plt.legend()

plt.savefig("/content/drive/MyDrive/GD/ROC_Curve_All_Models.png", dpi=300)

plt.show()

In [ ]:
plt.savefig("/content/drive/MyDrive/GD/ROC_Curve_All_Models.png", dpi=300)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/GD/glaucoma_features_CLINICAL_LABEL.csv")

# Remove non-numeric columns if needed
df = df.drop(columns=["dataset","image_index"], errors='ignore')

# Compute correlation matrix
corr_matrix = df.corr()

# Print correlation values
print(corr_matrix)

# Plot heatmap
plt.figure(figsize=(8,6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Feature Correlation Heatmap - Glaucoma Dataset")

plt.show()

In [ ]:
plt.savefig("/content/drive/MyDrive/GD/correlation_heatmap.png", dpi=300)

In [ ]:
# ===============================
# Import libraries
# ===============================
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier # Removed AdaBoostClassifier
# Removed as per user request
# from sklearn.svm import SVC
# from sklearn.linear_model import LogisticRegression
# from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
# Removed as per user request
# from catboost import CatBoostClassifier


# ===============================
# Train Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# ===============================
# Define Models
# ===============================
models = {

    "Random Forest": RandomForestClassifier(random_state=42),

    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),

    "LightGBM": LGBMClassifier(random_state=42)
    # Removed AdaBoost

}


# ===============================
# Confusion Matrix for each model
# ===============================
for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Normal","Glaucoma"],
        yticklabels=["Normal","Glaucoma"]
    )

    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    plt.show()

In [ ]:
import pandas as pd

from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier # Removed AdaBoostClassifier
# Removed as per user request
# from sklearn.svm import SVC
# from sklearn.linear_model import LogisticRegression
# from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
# Removed as per user request
# from catboost import CatBoostClassifier


# Define models
models = {

    "Random Forest": RandomForestClassifier(),

    "Decision Tree": DecisionTreeClassifier(),

    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss"),

    "LightGBM": LGBMClassifier()
    # Removed AdaBoost
}


# Create results list
results = []

for name, model in models.items():

    score_5 = cross_val_score(model, X_selected, y, cv=5).mean()

    score_10 = cross_val_score(model, X_selected, y, cv=10).mean()

    score_15 = cross_val_score(model, X_selected, y, cv=15).mean()

    score_20 = cross_val_score(model, X_selected, y, cv=20).mean()

    results.append([
        name,
        score_5,
        score_10,
        score_15,
        score_20
    ])


# Convert to dataframe
cv_table = pd.DataFrame(
    results,
    columns=[
        "Model",
        "5-Fold CV Accuracy",
        "10-Fold CV Accuracy",
        "15-Fold CV Accuracy",
        "20-Fold CV Accuracy"
    ]
)


# Show table
print(cv_table)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/GD/glaucoma_features_CLINICAL_LABEL.csv")

# Drop non-numeric columns
df = df.drop(columns=["dataset","image_index"], errors="ignore")

plt.figure(figsize=(10,6))

sns.boxplot(data=df)

plt.title("Box Plot of Glaucoma Features")

plt.xlabel("Features")
plt.ylabel("Values")

plt.xticks(rotation=45)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

features = df.columns

plt.figure(figsize=(12,8))

for i, feature in enumerate(features):

    plt.subplot(3,3,i+1)

    sns.histplot(df[feature], kde=True)

    plt.title(feature)

plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier # Removed AdaBoostClassifier
# Removed as per user request
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
# Removed as per user request
# from catboost import CatBoostClassifier


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced,
    y_balanced,
    test_size=0.3,
    random_state=42
)


# Define models
models = {

    "Random Forest": RandomForestClassifier(),

    "Decision Tree": DecisionTreeClassifier(),

    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss"),

    "LightGBM": LGBMClassifier()
    # Removed AdaBoost

}


results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:,1]


    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred)

    recall = recall_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)

    auc = roc_auc_score(y_test, y_prob)

    logloss = log_loss(y_test, y_prob)


    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        auc,
        logloss
    ])


# Create dataframe
metrics_table = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "AUC Score",
        "Log Loss"
    ]
)

print(metrics_table)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Melt the DataFrame for easier plotting
cv_table_melted = cv_table.melt(id_vars='Model',
                                var_name='CV Type',
                                value_name='Accuracy')

plt.figure(figsize=(12, 7))
sns.barplot(x='CV Type', y='Accuracy', hue='Model', data=cv_table_melted, palette='viridis')

plt.title('Cross-Validation Accuracy Across Different Models and Folds')
plt.xlabel('Cross-Validation Type')
plt.ylabel('Average Accuracy')
plt.ylim(0.95, 1.0) # Set y-axis limits to better visualize differences
plt.xticks(rotation=45, ha='right')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier # Removed AdaBoostClassifier
# Removed as per user request
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier # Added LightGBM

# Train-test split (using X_balanced and y_balanced from previous steps)
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced,
    y_balanced,
    test_size=0.3,
    random_state=42
)

# Define models
models = {
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss"),
    "LightGBM": LGBMClassifier() # Added LightGBM
    # Removed AdaBoost
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        auc
    ])

# Create dataframe
metrics_table = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "AUC Score"
    ]
)

# Melt the metrics_table for easier plotting
metrics_melted = metrics_table.melt(id_vars='Model',
                                    var_name='Metric',
                                    value_name='Score')

plt.figure(figsize=(16, 10))
sns.barplot(x='Metric', y='Score', hue='Model', data=metrics_melted, palette='viridis')

plt.title('Performance Evaluation Across Models')
plt.xlabel('Metric')
plt.ylabel('Score')
plt.ylim(0.9, 1.02) # Metrics typically range from 0 to 1
plt.xticks(rotation=45, ha='right')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
# from sklearn.svm import SVC # Removed as per user request
# from sklearn.neighbors import KNeighborsClassifier # Removed as per user request
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
# from catboost import CatBoostClassifier # Removed CatBoostClassifier

# --- 1. Prepare Data Without Dragonfly Feature Selection (but dropping 'VCDR') ---
# X and y from the current kernel state already represent the dataset
# after dropping 'dataset', 'image_index', 'label', and 'VCDR'.
# So, we use X and y as X_no_dfs and y_no_dfs.
X_no_dfs_original = X # This is the X before Dragonfly selection
y_no_dfs_original = y # This is the y before SMOTE

# Apply SMOTE to the data without Dragonfly feature selection
smote_no_dfs = SMOTE(random_state=42)
X_balanced_no_dfs, y_balanced_no_dfs = smote_no_dfs.fit_resample(X_no_dfs_original, y_no_dfs_original)

print("Balanced dataset (without DFS) distribution:")
print(pd.Series(y_balanced_no_dfs).value_counts())

# --- 2. Calculate Metrics for Models Without Dragonfly Feature Selection ---

# Train-test split for the 'without DFS' balanced data
X_train_no_dfs, X_test_no_dfs, y_train_no_dfs, y_test_no_dfs = train_test_split(
    X_balanced_no_dfs,
    y_balanced_no_dfs,
    test_size=0.3,
    random_state=42
)

# Define models (re-using the dictionary from the previous step ensures consistency)
models_for_comparison = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),
    "LightGBM": LGBMClassifier(random_state=42)
}

results_no_dfs = []

for name, model_instance in models_for_comparison.items():
    model_instance.fit(X_train_no_dfs, y_train_no_dfs)
    y_pred_no_dfs = model_instance.predict(X_test_no_dfs)
    y_prob_no_dfs = model_instance.predict_proba(X_test_no_dfs)[:,1]

    accuracy = accuracy_score(y_test_no_dfs, y_pred_no_dfs)
    precision = precision_score(y_test_no_dfs, y_pred_no_dfs)
    recall = recall_score(y_test_no_dfs, y_pred_no_dfs)
    f1 = f1_score(y_test_no_dfs, y_pred_no_dfs)
    auc = roc_auc_score(y_test_no_dfs, y_prob_no_dfs)

    results_no_dfs.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        auc
    ])

metrics_table_no_dfs = pd.DataFrame(
    results_no_dfs,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "AUC Score"]
)

# --- 3. Combine and Plot Performance ---

# Add a 'Feature Selection' column to both dataframes
metrics_table['Feature Selection'] = 'With DFS'
metrics_table_no_dfs['Feature Selection'] = 'Without DFS'

# Concatenate the two metrics tables
combined_metrics_table = pd.concat([metrics_table, metrics_table_no_dfs])

# Melt the combined table for plotting
combined_metrics_melted = combined_metrics_table.melt(id_vars=['Model', 'Feature Selection'],
                                                      var_name='Metric',
                                                      value_name='Score')

# Filter for Accuracy data only
accuracy_data = combined_metrics_melted[combined_metrics_melted['Metric'] == 'Accuracy'].copy()

plt.figure(figsize=(12, 7))
sns.lineplot(
    data=accuracy_data,
    x='Model',
    y='Score',
    hue='Feature Selection',
    marker='o',
    palette='viridis'
)

plt.title('ML Model Accuracy: With vs. Without Dragonfly Feature Selection', fontsize=16)
plt.xlabel('ML Model', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.ylim(0.9, 1.0) # Set a reasonable y-limit for accuracy scores
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Feature Selection', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Filter the melted DataFrame to include only 'Accuracy' scores
accuracy_data = combined_metrics_melted[combined_metrics_melted['Metric'] == 'Accuracy']

# Create a single line plot showing both 'With DFS' and 'Without DFS' on the same graph
g = sns.catplot(
    data=accuracy_data,
    x='Model',
    y='Score',
    hue='Feature Selection', # Use Feature Selection as hue to differentiate lines
    kind='point',
    palette='viridis',
    height=6, aspect=2, # Adjusted height and aspect for a single wider plot
    linestyles='-',
    marker='o',
    sharey=True
)

g.set_axis_labels('ML Model', 'Accuracy Score')
g.fig.suptitle('ML Model Accuracy: With vs. Without Dragonfly Feature Selection', fontsize=16, y=1.02)
g.set(ylim=(0.9, 1.0)) # Set a reasonable y-limit for accuracy scores
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
plt.tight_layout(rect=[0, 0, 1, 0.98]) # Adjust layout to prevent suptitle overlap
plt.show()

## SHAP (SHapley Additive exPlanations) for Model Explainability

SHAP (SHapley Additive exPlanations) is a game theory approach to explain the output of any machine learning model. It connects optimal credit allocation with local explanations using Shapley values from game theory. SHAP values tell us how to fairly distribute the 'credit' for a prediction among the input features. This allows us to understand the impact of each feature on the model's output.

We will use the `shap` library to explain the predictions of the XGBoost classifier, which was one of your top-performing models.

In [ ]:
import shap
import xgboost as xgb
import numpy as np
import pandas as pd

# Ensure X_train, X_test, y_train, y_test from the balanced and selected data are used
# These were derived from X_balanced, y_balanced after train_test_split
# from a previous cell (e.g., egKa_RDWVmqX or 5f99e5d3)

# Re-initialize and train the XGBoost model to ensure it's fresh for explainability
# using the X_train and y_train from the balanced and feature-selected dataset.

xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
xgb_model.fit(X_train, y_train)

# Create a TreeExplainer for tree-based models
# For other models, you might use shap.KernelExplainer or shap.DeepExplainer
explainer = shap.TreeExplainer(xgb_model)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test)

# Get the feature names for the selected features
# feature_names was defined before X was converted to numpy array (cell e8nAf7gI_g9W)
# selected_features was defined in cell 1ceC38Q4_GoX
current_feature_names = feature_names[selected_features].tolist()

print("SHAP values calculated successfully. Using features:", current_feature_names)


### SHAP Summary Plot

The summary plot combines feature importance with feature effects. Each point on the plot is a Shapley value for an instance and a feature. The position on the y-axis is determined by the feature, and on the x-axis by the Shapley value. The color represents the feature's value (red high, blue low). Overlapping points are jittered on the y-axis, and the density of points is shown by the gray shading.

In [ ]:
import matplotlib.pyplot as plt

# Summary plot: combines feature importance and feature effects
shap.summary_plot(shap_values, X_test, feature_names=current_feature_names, show=False)
plt.title('SHAP Summary Plot for XGBoost Classifier')
plt.tight_layout()
plt.show()


### SHAP Dependence Plot

A SHAP dependence plot shows how the value of a single feature affects the prediction of the model. It's similar to a partial dependence plot, but it's conditioned on SHAP values. The x-axis represents the feature value, and the y-axis is the SHAP value. The color of the points often represents an interacting feature, which helps to identify interaction effects.

In [ ]:
# Choose the most impactful feature from the summary plot (e.g., the top feature)
# For example, if 'Disc_Vertical_Diameter' is the top feature at index 3 within selected features
# You might need to adjust this based on the actual summary plot output.

# Assuming Cup_Area (index 0) or Disc_Area (index 1) are often important from common sense
# Let's pick 'Cup_Area' which is at index 0 in current_feature_names for X_selected
feature_to_explain_idx = 0 # Corresponds to 'Cup_Area' in current_feature_names

plt.figure(figsize=(8, 6))
shap.dependence_plot(
    feature_to_explain_idx, shap_values, X_test,
    feature_names=current_feature_names, show=False
)
plt.title(f'SHAP Dependence Plot for {current_feature_names[feature_to_explain_idx]}')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier # Removed AdaBoostClassifier
# Removed as per user request
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier # Added LightGBM

# --- 1. Data Loading and Initial Preprocessing --- (Replicating from previous cells)
path = "/content/drive/MyDrive/GD/glaucoma_features_CLINICAL_LABEL.csv"
df = pd.read_csv(path)
df.columns = df.columns.str.strip()

# Define X and y BEFORE any feature selection, after dropping specific columns
X_initial_df = df.drop(["dataset","image_index","label","VCDR"], axis=1)
y_initial = df["label"]

# Save feature names before converting to numpy
feature_names = X_initial_df.columns

# Convert to numpy arrays - this is our X_no_dfs_original
X_raw_numpy = X_initial_df.values
y_raw_numpy = y_initial.values


# --- 2. Dragonfly Feature Selection Function --- (Replicating from cell Q12JE91P_GlC)
def dragonfly_feature_selection(X_input, y_input, population_size=20, iterations=30):
    n_features = X_input.shape[1]
    population = np.random.randint(0, 2, (population_size, n_features))
    best_solution = None
    best_score = -1.0 # Initialize with a value lower than any possible accuracy score (0 to 1)

    for iteration in range(iterations):
        fitness = []
        for dragonfly in population:
            if np.sum(dragonfly) < 4: # Force minimum 4 features
                fitness.append(0)
                continue
            selected = np.where(dragonfly == 1)[0]
            if len(selected) == 0:
                fitness.append(0)
                continue

            X_subset = X_input[:, selected]
            model = RandomForestClassifier(n_estimators=100)
            score = cross_val_score(
                model,
                X_subset,
                y_input,
                cv=5,
                scoring="accuracy"
            ).mean()
            fitness.append(score)

        fitness = np.array(fitness)
        best_idx = np.argmax(fitness)
        if fitness[best_idx] > best_score:
            best_score = fitness[best_idx]
            best_solution = population[best_idx].copy()

        for i in range(population_size):
            step = best_solution - population[i]
            sigmoid = 1 / (1 + np.exp(-step))
            rand = np.random.rand(n_features)
            population[i] = np.where(rand < sigmoid, 1, 0)

    selected_features_indices = np.where(best_solution == 1)[0]
    return selected_features_indices


# --- 3. Apply Dragonfly Feature Selection to get X_selected_dfs ---
selected_features_indices = dragonfly_feature_selection(X_raw_numpy, y_raw_numpy)
X_selected_dfs = X_raw_numpy[:, selected_features_indices]


# --- 4. SMOTE for 'With DFS' path ---
smote_dfs = SMOTE(random_state=42)
X_balanced_dfs, y_balanced_dfs = smote_dfs.fit_resample(X_selected_dfs, y_raw_numpy)

# --- 5. SMOTE for 'Without DFS' path ---
smote_no_dfs = SMOTE(random_state=42)
X_balanced_no_dfs, y_balanced_no_dfs = smote_no_dfs.fit_resample(X_raw_numpy, y_raw_numpy)


# --- 6. Define Models for Evaluation ---
models_for_evaluation = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),
    "LightGBM": LGBMClassifier(random_state=42)
    # Removed AdaBoost
}


# --- 7. Calculate Metrics for 'With DFS' (metrics_table) ---
X_train_dfs, X_test_dfs, y_train_dfs, y_test_dfs = train_test_split(
    X_balanced_dfs,
    y_balanced_dfs,
    test_size=0.3,
    random_state=42
)

results_dfs = []
for name, model_instance in models_for_evaluation.items():
    model_instance.fit(X_train_dfs, y_train_dfs)
    y_pred_dfs = model_instance.predict(X_test_dfs)
    y_prob_dfs = model_instance.predict_proba(X_test_dfs)[:,1]

    accuracy = accuracy_score(y_test_dfs, y_pred_dfs)
    precision = precision_score(y_test_dfs, y_pred_dfs)
    recall = recall_score(y_test_dfs, y_pred_dfs)
    f1 = f1_score(y_test_dfs, y_pred_dfs)
    auc = roc_auc_score(y_test_dfs, y_prob_dfs)

    results_dfs.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        auc
    ])
metrics_table = pd.DataFrame(
    results_dfs,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "AUC Score"]
)


# --- 8. Calculate Metrics for 'Without DFS' (metrics_table_no_dfs) ---
X_train_no_dfs, X_test_no_dfs, y_train_no_dfs, y_test_no_dfs = train_test_split(
    X_balanced_no_dfs,
    y_balanced_no_dfs,
    test_size=0.3,
    random_state=42
)

results_no_dfs = []
for name, model_instance in models_for_evaluation.items():
    model_instance.fit(X_train_no_dfs, y_train_no_dfs)
    y_pred_no_dfs = model_instance.predict(X_test_no_dfs)
    y_prob_no_dfs = model_instance.predict_proba(X_test_no_dfs)[:,1]

    accuracy = accuracy_score(y_test_no_dfs, y_pred_no_dfs)
    precision = precision_score(y_test_no_dfs, y_pred_no_dfs)
    recall = recall_score(y_test_no_dfs, y_pred_no_dfs)
    f1 = f1_score(y_test_no_dfs, y_pred_no_dfs)
    auc = roc_auc_score(y_test_no_dfs, y_prob_no_dfs)

    results_no_dfs.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        auc
    ])
metrics_table_no_dfs = pd.DataFrame(
    results_no_dfs,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "AUC Score"]
)


# --- 9. Combine and Plot Performance ---
metrics_table['Feature Selection'] = 'With DFS'
metrics_table_no_dfs['Feature Selection'] = 'Without DFS'
combined_metrics_table = pd.concat([metrics_table, metrics_table_no_dfs])
combined_metrics_melted = combined_metrics_table.melt(id_vars=['Model', 'Feature Selection'],
                                                      var_name='Metric',
                                                      value_name='Score')

# Filter the melted DataFrame to include only 'Accuracy' scores
accuracy_data = combined_metrics_melted[combined_metrics_melted['Metric'] == 'Accuracy'].copy()

# Create a new column 'Research Work' by combining 'Model' and 'Feature Selection'
accuracy_data['Research Work'] = accuracy_data['Model'] + ' (' + accuracy_data['Feature Selection'] + ')'

plt.figure(figsize=(16, 8))
sns.barplot(
    x='Research Work',
    y='Score',
    data=accuracy_data,
    palette='viridis',
    edgecolor='black',
    linewidth=0.5
)

plt.title('Accuracy by Model and Feature Selection (Research Work)', fontsize=16)
plt.xlabel('Research Work (Model and Feature Selection Type)', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.ylim(0.8, 1.0) # Set a reasonable y-limit for accuracy scores
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()

## LIME (Local Interpretable Model-agnostic Explanations) for Model Explainability

LIME is a technique that explains the predictions of any classifier or regressor in an interpretable and faithful manner, by approximating the model locally with an interpretable model. It provides insights into why a model made a specific prediction for a particular data instance, making it especially useful for understanding individual cases.

We will use the `lime` library to explain a single prediction made by our XGBoost classifier.

In [ ]:
!pip install lime
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import numpy as np

# Ensure the XGBoost model is trained (from the previous SHAP cell)
# xgb_model is available from the SHAP setup cell

# Define the predict_proba function required by LIME
# This function should take a 2D array of instances and return class probabilities
def predict_proba_func(X):
    return xgb_model.predict_proba(X)

# Get feature names for the explainer
# current_feature_names was defined in the SHAP setup cell

# Initialize LimeTabularExplainer
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train, # Use the training data to learn feature statistics
    feature_names=current_feature_names,
    class_names=['Normal', 'Glaucoma'], # Assuming 0: Normal, 1: Glaucoma
    mode='classification'
)

# Choose an instance from the test set to explain (e.g., the first instance)
instance_to_explain_idx = 0
instance_to_explain = X_test[instance_to_explain_idx]

# Generate explanation for the instance
explanation = explainer_lime.explain_instance(
    data_row=instance_to_explain,
    predict_fn=predict_proba_func,
    num_features=len(current_feature_names) # Explain all features
)

print(f"LIME Explanation for Instance {instance_to_explain_idx}:")
print(f"Predicted class: {xgb_model.predict(instance_to_explain.reshape(1, -1))[0]} (Actual: {y_test[instance_to_explain_idx]})\n")

# Visualize the explanation
explanation.show_in_notebook(show_table=True, show_all=False)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier # Removed AdaBoostClassifier
# Removed as per user request
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier # Added LightGBM

# --- 1. Data Loading and Initial Preprocessing --- (Replicating from previous cells)
path = "/content/drive/MyDrive/GD/glaucoma_features_CLINICAL_LABEL.csv"
df = pd.read_csv(path)
df.columns = df.columns.str.strip()

# Define X and y BEFORE any feature selection, after dropping specific columns
X_initial_df = df.drop(["dataset","image_index","label","VCDR"], axis=1)
y_initial = df["label"]

# Save feature names before converting to numpy
feature_names = X_initial_df.columns

# Convert to numpy arrays - this is our X_no_dfs_original
X_raw_numpy = X_initial_df.values
y_raw_numpy = y_initial.values


# --- 2. Dragonfly Feature Selection Function --- (Replicating from cell Q12JE91P_GlC)
def dragonfly_feature_selection(X_input, y_input, population_size=20, iterations=30):
    n_features = X_input.shape[1]
    population = np.random.randint(0, 2, (population_size, n_features))
    best_solution = None
    best_score = -1.0 # Initialize with a value lower than any possible accuracy score (0 to 1)

    for iteration in range(iterations):
        fitness = []
        for dragonfly in population:
            if np.sum(dragonfly) < 4: # Force minimum 4 features
                fitness.append(0)
                continue
            selected = np.where(dragonfly == 1)[0]
            if len(selected) == 0:
                fitness.append(0)
                continue

            X_subset = X_input[:, selected]
            model = RandomForestClassifier(n_estimators=100)
            score = cross_val_score(
                model,
                X_subset,
                y_input,
                cv=5,
                scoring="accuracy"
            ).mean()
            fitness.append(score)

        fitness = np.array(fitness)
        best_idx = np.argmax(fitness)
        if fitness[best_idx] > best_score:
            best_score = fitness[best_idx]
            best_solution = population[best_idx].copy()

        for i in range(population_size):
            step = best_solution - population[i]
            sigmoid = 1 / (1 + np.exp(-step))
            rand = np.random.rand(n_features)
            population[i] = np.where(rand < sigmoid, 1, 0)

    selected_features_indices = np.where(best_solution == 1)[0]
    return selected_features_indices


# --- 3. Apply Dragonfly Feature Selection to get X_selected_dfs ---
selected_features_indices = dragonfly_feature_selection(X_raw_numpy, y_raw_numpy)
X_selected_dfs = X_raw_numpy[:, selected_features_indices]


# --- 4. SMOTE for 'With DFS' path ---
smote_dfs = SMOTE(random_state=42)
X_balanced_dfs, y_balanced_dfs = smote_dfs.fit_resample(X_selected_dfs, y_raw_numpy)

# --- 5. SMOTE for 'Without DFS' path ---
smote_no_dfs = SMOTE(random_state=42)
X_balanced_no_dfs, y_balanced_no_dfs = smote_no_dfs.fit_resample(X_raw_numpy, y_raw_numpy)


# --- 6. Define Models for Evaluation ---
models_for_evaluation = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),
    "LightGBM": LGBMClassifier(random_state=42)
    # Removed AdaBoost
}


# --- 7. Calculate Metrics for 'With DFS' (metrics_table) ---
X_train_dfs, X_test_dfs, y_train_dfs, y_test_dfs = train_test_split(
    X_balanced_dfs,
    y_balanced_dfs,
    test_size=0.3,
    random_state=42
)

results_dfs = []
for name, model_instance in models_for_evaluation.items():
    model_instance.fit(X_train_dfs, y_train_dfs)
    y_pred_dfs = model_instance.predict(X_test_dfs)
    y_prob_dfs = model_instance.predict_proba(X_test_dfs)[:,1]

    accuracy = accuracy_score(y_test_dfs, y_pred_dfs)
    precision = precision_score(y_test_dfs, y_pred_dfs)
    recall = recall_score(y_test_dfs, y_pred_dfs)
    f1 = f1_score(y_test_dfs, y_pred_dfs)
    auc = roc_auc_score(y_test_dfs, y_prob_dfs)

    results_dfs.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        auc
    ])
metrics_table = pd.DataFrame(
    results_dfs,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "AUC Score"]
)


# --- 8. Calculate Metrics for 'Without DFS' (metrics_table_no_dfs) ---
X_train_no_dfs, X_test_no_dfs, y_train_no_dfs, y_test_no_dfs = train_test_split(
    X_balanced_no_dfs,
    y_balanced_no_dfs,
    test_size=0.3,
    random_state=42
)

results_no_dfs = []
for name, model_instance in models_for_evaluation.items():
    model_instance.fit(X_train_no_dfs, y_train_no_dfs)
    y_pred_no_dfs = model_instance.predict(X_test_no_dfs)
    y_prob_no_dfs = model_instance.predict_proba(X_test_no_dfs)[:,1]

    accuracy = accuracy_score(y_test_no_dfs, y_pred_no_dfs)
    precision = precision_score(y_test_no_dfs, y_pred_no_dfs)
    recall = recall_score(y_test_no_dfs, y_pred_no_dfs)
    f1 = f1_score(y_test_no_dfs, y_pred_no_dfs)
    auc = roc_auc_score(y_test_no_dfs, y_prob_no_dfs)

    results_no_dfs.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        auc
    ])
metrics_table_no_dfs = pd.DataFrame(
    results_no_dfs,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "AUC Score"]
)


# --- 9. Combine and Plot Performance ---
metrics_table['Feature Selection'] = 'With DFS'
metrics_table_no_dfs['Feature Selection'] = 'Without DFS'
combined_metrics_table = pd.concat([metrics_table, metrics_table_no_dfs])
combined_metrics_melted = combined_metrics_table.melt(id_vars=['Model', 'Feature Selection'],
                                                      var_name='Metric',
                                                      value_name='Score')

# Filter the melted DataFrame to include only 'Accuracy' scores
accuracy_data = combined_metrics_melted[combined_metrics_melted['Metric'] == 'Accuracy'].copy()

# Create a new column 'Research Work' by combining 'Model' and 'Feature Selection'
accuracy_data['Research Work'] = accuracy_data['Model'] + ' (' + accuracy_data['Feature Selection'] + ')'

plt.figure(figsize=(16, 8))
sns.barplot(
    x='Research Work',
    y='Score',
    data=accuracy_data,
    palette='viridis',
    edgecolor='black',
    linewidth=0.5
)

plt.title('Accuracy by Model and Feature Selection (Research Work)', fontsize=16)
plt.xlabel('Research Work (Model and Feature Selection Type)', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.ylim(0.8, 1.0) # Set a reasonable y-limit for accuracy scores
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()